# 🏭 GarudaChip — Verilator Lint → **step-by-step** LibreLane → GDSII + signoff reports

Mirrors the LibreLane Colab notebook: every implementation step runs in its **own cell**
with `display(step)`, instead of one `librelane config.json` call. It ends by **rendering
the GDSII layout image** and printing the **DRC / LVS / antenna / timing** sign-off reports
— the real tape-out checks, not just a netlist.

Flow: lint → `Config.interactive` → Synthesis → Floorplan → Tap/Endcap → I/O → PDN →
Global/Detailed Placement → CTS → Global/Detailed Routing → Fill → RCX → STA → **StreamOut
(GDS)** → **DRC** → SPICE extract → **LVS** → 🖼️ GDS image + 📋 reports.

**Design:** `examples/verilog_designs/generated_pwm_generator/pwm_generator.v` (the simplest
clean block). **PDK:** `gf180mcuD` at `~/.ciel`.

> ⚠️ **Run this notebook in LibreLane's Python environment** (the one where `import librelane`
> works — the Nix-installed LibreLane), NOT the GarudaChip `uv` venv. Easiest:
> `nix run github:librelane/librelane -- --help` to confirm it's installed, then launch
> Jupyter from that env (or register that Python as a Jupyter kernel and select it here).
> The step IDs below are exactly those of the official LibreLane Colab notebook.


## 0 · Locate tools, PDK, and stage the design

In [ ]:
import os, glob, shutil, subprocess
from pathlib import Path
from IPython.display import display, Image, Markdown

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "examples" / "verilog_designs").is_dir()), Path.cwd())

PDK       = os.environ.setdefault("PDK", "gf180mcuD")
PDK_ROOT  = os.environ.setdefault("PDK_ROOT", os.path.expanduser("~/.ciel"))
print("Repo     :", REPO)
print("PDK      :", PDK, "@", PDK_ROOT, "(exists)" if Path(PDK_ROOT, PDK).exists() else "(MISSING)")

def find_verilator():
    return (shutil.which("verilator") or os.getenv("VERILATOR_BIN")
            or (sorted(glob.glob("/nix/store/*verilator*/bin/verilator")) or [None])[-1])
VERILATOR = find_verilator()
print("verilator:", VERILATOR or "NOT FOUND")
import librelane; print("librelane:", librelane.__version__)

SRC  = REPO / "examples" / "verilog_designs" / "generated_pwm_generator"
TOP  = "pwm_generator"
WORK = REPO / "output" / "nb_step_harden"
RTL  = WORK / "rtl"
if WORK.exists(): shutil.rmtree(WORK)
RTL.mkdir(parents=True, exist_ok=True)
shutil.copy(SRC / f"{TOP}.v", RTL / f"{TOP}.v")
print("staged   ->", RTL / f"{TOP}.v")

## 1 · Verilator structural lint (the gate before hardening)

In [ ]:
assert VERILATOR, "verilator not found"
cmd = [VERILATOR, "--lint-only", "-Wno-fatal",
       "--Werror-MULTIDRIVEN", "--Werror-LATCH", "--Werror-UNOPTFLAT",
       f"-I{RTL}", "--top-module", TOP, str(RTL / f"{TOP}.v")]   # NOTE: attached -I (no space)
p = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
out = (p.stdout + p.stderr).strip()
print(out or "(clean)")
assert p.returncode == 0 and "%Error" not in out, "LINT FAILED — fix RTL before hardening"
print("\n✅ Lint clean — structurally sound for PnR.")

## 2 · LibreLane configuration (`Config.interactive`)

Like the Colab, `Config.interactive` propagates the config to every step we create below.
PWM is tiny, so we give an explicit small die and a denser PDN.


In [ ]:
from librelane.config import Config
Config.interactive(
    TOP,
    PDK=PDK,
    PDK_ROOT=PDK_ROOT,
    CLOCK_PORT="clk",
    CLOCK_NET="clk",
    CLOCK_PERIOD=24,
    FP_SIZING="absolute",
    DIE_AREA=[0, 0, 150, 150],
    FP_CORE_UTIL=35,
    PL_TARGET_DENSITY_PCT=45,
    PRIMARY_GDSII_STREAMOUT_TOOL="klayout",
    # auto-generated RTL lint is informational here (we already gated on Verilator)
    ERROR_ON_LINTER_ERRORS=False, ERROR_ON_LINTER_WARNINGS=False,
)
from librelane.steps import Step
from librelane.state import State
print("✅ interactive config set for", TOP)

## 3 · Synthesis — RTL → standard-cell netlist

In [ ]:
Synthesis = Step.factory.get("Yosys.Synthesis")
synthesis = Synthesis(VERILOG_FILES=[str(RTL / f"{TOP}.v")], state_in=State())
synthesis.start()
display(synthesis)

## 4 · Floorplan — die/core dimensions + the placement grid

In [ ]:
Floorplan = Step.factory.get("OpenROAD.Floorplan")
floorplan = Floorplan(state_in=synthesis.state_out)
floorplan.start()
display(floorplan)

## 5 · Tap / End-cap insertion (well-tie + boundary cells)

In [ ]:
TapEndcap = Step.factory.get("OpenROAD.TapEndcapInsertion")
tap = TapEndcap(state_in=floorplan.state_out)
tap.start()
display(tap)

## 6 · I/O pin placement (top-level ports at the edges)

In [ ]:
IOPlacement = Step.factory.get("OpenROAD.IOPlacement")
ioplace = IOPlacement(state_in=tap.state_out)
ioplace.start()
display(ioplace)

## 7 · Power Distribution Network (VDD/VSS straps + rails)

In [ ]:
GeneratePDN = Step.factory.get("OpenROAD.GeneratePDN")
pdn = GeneratePDN(state_in=ioplace.state_out, FP_PDN_VWIDTH=2, FP_PDN_HWIDTH=2,
                  FP_PDN_VPITCH=30, FP_PDN_HPITCH=30)
pdn.start()
display(pdn)

## 8 · Global placement (fuzzy cell locations, min wirelength)

In [ ]:
GlobalPlacement = Step.factory.get("OpenROAD.GlobalPlacement")
gpl = GlobalPlacement(state_in=pdn.state_out)
gpl.start()
display(gpl)

## 9 · Detailed placement (legalize to the grid)

In [ ]:
DetailedPlacement = Step.factory.get("OpenROAD.DetailedPlacement")
dpl = DetailedPlacement(state_in=gpl.state_out)
dpl.start()
display(dpl)

## 10 · Clock Tree Synthesis (balanced clock buffers, low skew)

In [ ]:
CTS = Step.factory.get("OpenROAD.CTS")
cts = CTS(state_in=dpl.state_out)
cts.start()
display(cts)

## 11 · Global routing (plan the wire routes — no display)

In [ ]:
GlobalRouting = Step.factory.get("OpenROAD.GlobalRouting")
grt = GlobalRouting(state_in=cts.state_out)
grt.start()

## 12 · Detailed routing (real metal wires — the longest step)

In [ ]:
DetailedRouting = Step.factory.get("OpenROAD.DetailedRouting")
drt = DetailedRouting(state_in=grt.state_out)
drt.start()
display(drt)

## 13 · Fill insertion (decap + fill the gaps)

In [ ]:
FillInsertion = Step.factory.get("OpenROAD.FillInsertion")
fill = FillInsertion(state_in=drt.state_out)
fill.start()
display(fill)

## 14 · Parasitic extraction (RCX → SPEF for timing)

In [ ]:
RCX = Step.factory.get("OpenROAD.RCX")
rcx = RCX(state_in=fill.state_out)
rcx.start()

## 15 · Post-PnR Static Timing Analysis (.lib / .sdf)

In [ ]:
STAPostPNR = Step.factory.get("OpenROAD.STAPostPNR")
sta = STAPostPNR(state_in=rcx.state_out)
sta.start()

## 16 · Stream-out → **GDSII** (the fabrication layout)

In [ ]:
StreamOut = Step.factory.get("KLayout.StreamOut")
gds = StreamOut(state_in=sta.state_out)
gds.start()
display(gds)

## 17 · 🖼️ Render and SHOW the GDSII layout image

In [ ]:
# find the streamed GDS in this step's state, render it to a PNG with KLayout, and show it
gds_path = None
for k, v in dict(gds.state_out).items():
    if isinstance(v, str) and v.endswith(".gds"):
        gds_path = v; break
gds_path = gds_path or (sorted(glob.glob(str(WORK / "**" / "*.gds"), recursive=True)) or [None])[-1]
print("GDS:", gds_path)

png = str(WORK / f"{TOP}_layout.png")
try:
    import klayout.lay as klay
    lv = klay.LayoutView(); lv.load_layout(gds_path, True); lv.max_hier(); lv.zoom_fit()
    lv.save_image(png, 1400, 1400)
    display(Markdown(f"### 🎉 {TOP} — GDSII layout")); display(Image(filename=png))
except Exception as e:
    print("KLayout render unavailable:", e, "\nGDS is at:", gds_path)

## 18 · Design Rule Check (DRC) — is the layout manufacturable?

In [ ]:
DRC = Step.factory.get("Magic.DRC")
drc = DRC(state_in=gds.state_out)
drc.start()
display(drc)
# show the DRC violation count + report
m = dict(getattr(drc, "metrics", {}) or {})
viol = m.get("magic__drc__count", m.get("magic__drc_error__count", "n/a"))
print(f"\n📋 DRC violations: {viol}")
for r in sorted(glob.glob(str(WORK / "**" / "*drc*.rpt"), recursive=True))[:1]:
    print("---", r, "---"); print(Path(r).read_text()[:1500])

## 19 · SPICE extraction (rebuild a netlist from the layout for LVS)

In [ ]:
SpiceExtraction = Step.factory.get("Magic.SpiceExtraction")
spx = SpiceExtraction(state_in=drc.state_out)
spx.start()
print("✅ extracted SPICE netlist from the GDS for LVS")

## 20 · Layout-vs-Schematic (LVS) — does the layout match the netlist?

In [ ]:
LVS = Step.factory.get("Netgen.LVS")
lvs = LVS(state_in=spx.state_out)
lvs.start()
display(lvs)
m = dict(getattr(lvs, "metrics", {}) or {})
print(f"\n📋 LVS errors: {m.get('netgen__lvs__errors__count', 'n/a')}")
for r in sorted(glob.glob(str(WORK / "**" / "*lvs*.rpt"), recursive=True))[:1]:
    print("---", r, "---"); print(Path(r).read_text()[:1500])

## 21 · 📋 Sign-off summary (all reports in one place)

In [ ]:
def metric(step, *keys, default="n/a"):
    m = dict(getattr(step, "metrics", {}) or {})
    for k in keys:
        if k in m: return m[k]
    return default

rows = [
    ("Die area (µm²)",      metric(floorplan, "design__die__area")),
    ("Std cells",           metric(synthesis, "design__instance__count__stdcell",
                                              "synthesis__instance__count")),
    ("Setup worst-slack (ns)", metric(sta, "timing__setup__ws")),
    ("Hold worst-slack (ns)",  metric(sta, "timing__hold__ws")),
    ("DRC violations",      metric(drc, "magic__drc__count", "magic__drc_error__count")),
    ("LVS errors",          metric(lvs, "netgen__lvs__errors__count")),
]
print(f"\n{'='*46}\n  {TOP} — TAPE-OUT SIGN-OFF\n{'='*46}")
for k, v in rows:
    print(f"  {k:26s} {v}")
print(f"{'='*46}")
print("\nFull per-step logs/reports are under:", WORK)